# 02 · Bronze → Silver

Conceitos praticados:
- **Deduplicação** (`dropDuplicates` por chave de negócio)
- **Tratamento de nulos** (nulos estruturais vs. nulos de negócio — nem todo nulo deve ser preenchido!)
- **Conversão de formatos** (string → timestamp, string → decimal/double)
- **Validações de qualidade de dados** com um log de resultados persistido em Delta

In [0]:
catalog = "olist_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

In [0]:
from pyspark.sql import functions as F, Row
from datetime import datetime

dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

## 1. customers
- Deduplicação por `customer_id` (renomeado para `id_consumidor`)
- Verificação de unicidade de `id_consumidor` antes de salvar (não utiliza `customer_unique_id`)
- Nomes de Estado e Cidade em MAIÚSCULAS (Upper Case)
- Nulos: descartamos linhas sem `customer_id` (nulo estrutural = registro inutilizável)

In [0]:
df_bronze_customers = spark.table(f"{bronze_schema}.customers")

df_silver_customers = (
    df_bronze_customers
    .dropDuplicates(["customer_id"])
    .filter(F.col("customer_id").isNotNull())
    .withColumn("customer_city", F.upper(F.trim(F.col("customer_city"))))
    .withColumn("customer_state", F.upper(F.trim(F.col("customer_state"))))
    .withColumn("customer_zip_code_prefix", F.trim(F.col("customer_zip_code_prefix")))
    .select("customer_id", "customer_unique_id", "customer_zip_code_prefix",
            "customer_city", "customer_state")
    .withColumnRenamed("customer_id", "id_consumidor")
    .withColumnRenamed("customer_unique_id", "id_cliente_unico")
    .withColumnRenamed("customer_zip_code_prefix", "prefixo_cep_cliente")
    .withColumnRenamed("customer_city", "cidade_cliente")
    .withColumnRenamed("customer_state", "estado_cliente")
)

dq_check("silver.customers", "id_consumidor não nulo", df_silver_customers, F.col("id_consumidor").isNotNull())
dq_check_unique("silver.customers", "id_consumidor único", df_silver_customers, ["id_consumidor"])
dq_check("silver.customers", "estado_cliente com 2 caracteres", df_silver_customers,
         F.length("estado_cliente") == 2)

df_silver_customers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.customers")

display(df_bronze_customers.limit(10))
display(df_silver_customers.limit(10))

[PASS] silver.customers | id_consumidor não nulo | 0/99441 linhas falharam
[PASS] silver.customers | id_consumidor único | 0 chaves duplicadas de 99441 linhas
[PASS] silver.customers | estado_cliente com 2 caracteres | 0/99441 linhas falharam


customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_name,customer_gender,customer_birth_date,customer_age,_ingested_at
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,Zoe Nogueira,F,1956-09-07,70,2026-09-15T01:06:57.073Z
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,Liam Viana,M,1974-09-23,52,2026-09-15T01:06:57.073Z
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,Diego Aparecida,M,1964-05-20,62,2026-09-15T01:06:57.073Z
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,Marcos Vinicius Azevedo,M,1967-01-14,59,2026-09-15T01:06:57.073Z
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,Srta. Juliana Siqueira,F,1950-08-01,76,2026-09-15T01:06:57.073Z
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC,Dra. Alice Lopes,F,1975-07-07,51,2026-09-15T01:06:57.073Z
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,sao paulo,SP,Maysa Vieira,F,1998-09-27,28,2026-09-15T01:06:57.073Z
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG,José Miguel Freitas,M,2004-01-08,22,2026-09-15T01:06:57.073Z
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR,Samuel Duarte,M,1991-08-22,35,2026-09-15T01:06:57.073Z
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG,Rael Pimenta,M,1995-11-12,31,2026-09-15T01:06:57.073Z


id_consumidor,id_cliente_unico,prefixo_cep_cliente,cidade_cliente,estado_cliente
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,FRANCA,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,SAO BERNARDO DO CAMPO,SP
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,SAO PAULO,SP
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,MOGI DAS CRUZES,SP
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,CAMPINAS,SP
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,JARAGUA DO SUL,SC
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,SAO PAULO,SP
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,TIMOTEO,MG
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,CURITIBA,PR
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,BELO HORIZONTE,MG


## 2. orders
- Deduplicação por `order_id`
- Conversão string → timestamp
- Tradução de `order_status` do inglês para português (delivered→entregue, invoiced→faturado, etc.)
- Colunas derivadas:
  - `tempo_entrega_dias`: diferença em dias entre entrega e compra
  - `tempo_entrega_estimado_dias`: diferença em dias entre entrega estimada e compra
  - `diferenca_entrega_dias`: diferença entre tempo real e estimado
  - `entrega_no_prazo`: "Sim" (no prazo), "Não" (atrasado), "Não Entregue" (pendente)
- Nulos **preservados intencionalmente**: `data_entrega_cliente` nulo pode significar
  apenas que o pedido ainda não foi entregue — preencher isso seria inventar um dado incorreto.

In [0]:
df_bronze_orders = spark.table(f"{bronze_schema}.orders")

df_silver_orders = (
    df_bronze_orders
    .dropDuplicates(["order_id"])
    .filter(F.col("order_id").isNotNull() & F.col("customer_id").isNotNull())
    .withColumn("order_status", F.lower(F.trim(F.col("order_status"))))
    .withColumn("order_status",
        F.when(F.col("order_status") == "delivered", "entregue")
        .when(F.col("order_status") == "invoiced", "faturado")
        .when(F.col("order_status") == "shipped", "enviado")
        .when(F.col("order_status") == "processing", "em processamento")
        .when(F.col("order_status") == "unavailable", "indisponível")
        .when(F.col("order_status") == "canceled", "cancelado")
        .when(F.col("order_status") == "created", "criado")
        .when(F.col("order_status") == "approved", "aprovado")
        .otherwise(F.col("order_status"))
    )
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp"))
    .withColumn("order_approved_at", F.to_timestamp("order_approved_at"))
    .withColumn("order_delivered_carrier_date", F.to_timestamp("order_delivered_carrier_date"))
    .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date"))
    .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date"))
    .select("order_id", "customer_id", "order_status", "order_purchase_timestamp",
            "order_approved_at", "order_delivered_carrier_date",
            "order_delivered_customer_date", "order_estimated_delivery_date")
    .withColumnRenamed("order_id", "id_pedido")
    .withColumnRenamed("customer_id", "id_cliente")
    .withColumnRenamed("order_status", "status_pedido")
    .withColumnRenamed("order_purchase_timestamp", "data_compra_pedido")
    .withColumnRenamed("order_approved_at", "data_aprovacao_pedido")
    .withColumnRenamed("order_delivered_carrier_date", "data_entrega_transportadora")
    .withColumnRenamed("order_delivered_customer_date", "data_entrega_cliente")
    .withColumnRenamed("order_estimated_delivery_date", "data_estimada_entrega")
    .withColumn("tempo_entrega_dias",
        F.datediff(F.col("data_entrega_cliente"), F.col("data_compra_pedido")))
    .withColumn("tempo_entrega_estimado_dias",
        F.datediff(F.col("data_estimada_entrega"), F.col("data_compra_pedido")))
    .withColumn("diferenca_entrega_dias",
        F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias"))
    .withColumn("entrega_no_prazo",
        F.when(F.col("data_entrega_cliente").isNull(), "Não Entregue")
        .when(F.col("diferenca_entrega_dias") <= 0, "Sim")
        .otherwise("Não")
    )
)

valid_status = ["entregue", "enviado", "cancelado", "indisponível",
                 "faturado", "em processamento", "criado", "aprovado"]

dq_check_unique("silver.orders", "id_pedido único", df_silver_orders, ["id_pedido"])
dq_check("silver.orders", "data_compra_pedido não nulo", df_silver_orders,
         F.col("data_compra_pedido").isNotNull())
dq_check("silver.orders", "status_pedido em domínio válido", df_silver_orders,
         F.col("status_pedido").isin(valid_status))

df_silver_orders.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.orders")

display(df_bronze_orders.limit(10))
display(df_silver_orders.limit(10))

[PASS] silver.orders | id_pedido único | 0 chaves duplicadas de 99441 linhas
[PASS] silver.orders | data_compra_pedido não nulo | 0/99441 linhas falharam
[PASS] silver.orders | status_pedido em domínio válido | 0/99441 linhas falharam


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,_ingested_at
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,2026-09-15T01:07:00.018Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,2026-09-15T01:07:00.018Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,2026-09-15T01:07:00.018Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,2026-09-15T01:07:00.018Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z,2026-09-15T01:07:00.018Z
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z,2026-09-15T01:07:00.018Z
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z,2026-09-15T01:07:00.018Z
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z,2026-09-15T01:07:00.018Z
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z,2026-09-15T01:07:00.018Z
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z,2026-09-15T01:07:00.018Z


id_pedido,id_cliente,status_pedido,data_compra_pedido,data_aprovacao_pedido,data_entrega_transportadora,data_entrega_cliente,data_estimada_entrega,tempo_entrega_dias,tempo_entrega_estimado_dias,diferenca_entrega_dias,entrega_no_prazo
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,entregue,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,8,16,-8,Sim
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,entregue,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,14,20,-6,Sim
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,entregue,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,9,27,-18,Sim
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,entregue,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,14,27,-13,Sim
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,entregue,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z,3,13,-10,Sim
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,entregue,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z,17,23,-6,Sim
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,faturado,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z,null,28,null,Não Entregue
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,entregue,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z,10,22,-12,Sim
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,entregue,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z,10,42,-32,Sim
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,entregue,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z,18,25,-7,Sim


## 3. products
- Deduplicação por `product_id` (renomeado para `id_produto`)
- Enriquecimento com a tradução da categoria (join pequeno)
- Mapeamento de colunas: `product_category_name`→`categoria_produto`, `product_weight_g`→`peso_produto_gramas`, etc.
- Nulos de negócio (ex.: categoria ausente) tratados com valor `"outros"` / `"others"`;
  nulos em dimensões físicas viram `0` para não quebrar cálculos posteriores.

In [0]:
df_bronze_products = spark.table(f"{bronze_schema}.products")
df_bronze_translation = spark.table(f"{bronze_schema}.product_category_translation")

df_silver_products = (
    df_bronze_products
    .dropDuplicates(["product_id"])
    .filter(F.col("product_id").isNotNull())
    .join(df_bronze_translation, on="product_category_name", how="left")
    .withColumn("product_category_name", F.coalesce(F.col("product_category_name"), F.lit("outros")))
    .withColumn("product_category_name_english",
                F.coalesce(F.col("product_category_name_english"), F.lit("others")))
    .withColumn("product_weight_g", F.col("product_weight_g").cast("double"))
    .withColumn("product_length_cm", F.col("product_length_cm").cast("double"))
    .withColumn("product_height_cm", F.col("product_height_cm").cast("double"))
    .withColumn("product_width_cm", F.col("product_width_cm").cast("double"))
    .fillna({"product_weight_g": 0, "product_length_cm": 0,
             "product_height_cm": 0, "product_width_cm": 0})
    .select("product_id", "product_category_name", "product_category_name_english",
            "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm")
    .withColumnRenamed("product_id", "id_produto")
    .withColumnRenamed("product_category_name", "categoria_produto")
    .withColumnRenamed("product_category_name_english", "categoria_produto_ingles")
    .withColumnRenamed("product_weight_g", "peso_produto_gramas")
    .withColumnRenamed("product_length_cm", "comprimento_centimetros")
    .withColumnRenamed("product_height_cm", "altura_centimetros")
    .withColumnRenamed("product_width_cm", "largura_centimetros")
)

dq_check_unique("silver.products", "id_produto único", df_silver_products, ["id_produto"])
dq_check("silver.products", "peso_produto_gramas não negativo", df_silver_products,
         F.col("peso_produto_gramas") >= 0)

df_silver_products.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.products")

display(df_bronze_products.limit(10))
display(df_silver_products.limit(10))

[PASS] silver.products | id_produto único | 0 chaves duplicadas de 32951 linhas
[PASS] silver.products | peso_produto_gramas não negativo | 0/32951 linhas falharam


product_id,product_name,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,_ingested_at
1e9e8ef04dbcff4541ed26657ea517e5,Perfume Premium,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,2026-09-15T01:07:04.742Z
3aa071139cb16b67ca9e5dea641aaa2f,Conjunto de Pincéis,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,2026-09-15T01:07:04.742Z
96bd76ec8810374ed1b65e291975717f,Barraca de Camping,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,2026-09-15T01:07:04.742Z
cef67bcfe19066a932b7673e239eb23d,Chupeta Premium,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,2026-09-15T01:07:04.742Z
9dc1a7de274444849c219cff195d0b71,Vassoura Mágica,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,2026-09-15T01:07:04.742Z
41d3672d4792049fa1779bb35283ed13,Violão Acústico,instrumentos_musicais,60.0,745.0,1.0,200.0,38.0,5.0,11.0,2026-09-15T01:07:04.742Z
732bd381ad09e530fe0a5f457d81becb,Almofada Decorativa Azul,cool_stuff,56.0,1272.0,4.0,18350.0,70.0,24.0,44.0,2026-09-15T01:07:04.742Z
2548af3e6e77a690cf3eb6368e9ab61e,Tapete Felpudo Plus,moveis_decoracao,56.0,184.0,2.0,900.0,40.0,8.0,40.0,2026-09-15T01:07:04.742Z
37cc742be07708b53a98702e77a21a02,Máquina de Lavar Ultra,eletrodomesticos,57.0,163.0,1.0,400.0,27.0,13.0,17.0,2026-09-15T01:07:04.742Z
8c92109888e8cdf9d66dc7e463025574,Boneca Ultra,brinquedos,36.0,1156.0,1.0,600.0,17.0,10.0,12.0,2026-09-15T01:07:04.742Z


id_produto,categoria_produto,categoria_produto_ingles,peso_produto_gramas,comprimento_centimetros,altura_centimetros,largura_centimetros
e1d1d22e9f8122a4ec1533b032c12562,ferramentas_jardim,garden_tools,2150.0,70.0,8.0,34.0
ce5b91848b91118daffb3af53b747475,esporte_lazer,sports_leisure,1388.0,34.0,9.0,31.0
1c6fb703c624b381a20f21f757694866,brinquedos,toys,725.0,22.0,17.0,15.0
4e04ffb7dd3739ecfc37de8927dd586c,papelaria,stationery,500.0,24.0,7.0,16.0
0992c6cba95a13bfa68ea7d5e22d478b,ferramentas_jardim,garden_tools,3850.0,29.0,12.0,24.0
32ee3f4e2493045726807808d7abbab6,cama_mesa_banho,bed_bath_table,1400.0,38.0,9.0,27.0
af16005fca813272caf59c432153949e,moveis_decoracao,furniture_decor,800.0,53.0,8.0,20.0
b2b3503e675abbc07e278151da072ed2,moveis_decoracao,furniture_decor,8500.0,75.0,25.0,55.0
acd427ee119d5c71c7a85ae488cb0a6a,fashion_bolsas_e_acessorios,fashion_bags_accessories,200.0,16.0,2.0,11.0
51fed4afb1b41d00ad9d25a73b7fbbc7,consoles_games,consoles_games,250.0,21.0,7.0,15.0


## 4. order_items
- Deduplicação pela chave composta `(order_id, order_item_id)` → `(id_pedido, id_item)`
- Conversão para `decimal(10,2)` (tipo correto para valores monetários — evita erro de ponto flutuante)
- Mapeamento: `price`→`preco`, `freight_value`→`preco_frete`
- Validação: preço deve ser positivo

In [0]:
df_bronze_items = spark.table(f"{bronze_schema}.order_items")

df_silver_items = (
    df_bronze_items
    .dropDuplicates(["order_id", "order_item_id"])
    .filter(F.col("order_id").isNotNull() & F.col("product_id").isNotNull())
    .withColumn("price", F.col("price").cast("decimal(10,2)"))
    .withColumn("freight_value", F.col("freight_value").cast("decimal(10,2)"))
    .withColumn("shipping_limit_date", F.to_timestamp("shipping_limit_date"))
    .filter(F.col("price") > 0)
    .select("order_id", "order_item_id", "product_id", "seller_id",
            "shipping_limit_date", "price", "freight_value")
    .withColumnRenamed("order_id", "id_pedido")
    .withColumnRenamed("order_item_id", "id_item")
    .withColumnRenamed("product_id", "id_produto")
    .withColumnRenamed("seller_id", "id_vendedor")
    .withColumnRenamed("shipping_limit_date", "data_limite_envio")
    .withColumnRenamed("price", "preco")
    .withColumnRenamed("freight_value", "preco_frete")
)

dq_check_unique("silver.order_items", "(id_pedido, id_item) único", df_silver_items,
                 ["id_pedido", "id_item"])
dq_check("silver.order_items", "preco > 0", df_silver_items, F.col("preco") > 0)

df_silver_items.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.order_items")

display(df_bronze_items.limit(10))
display(df_silver_items.limit(10))


[PASS] silver.order_items | (id_pedido, id_item) único | 0 chaves duplicadas de 112650 linhas
[PASS] silver.order_items | preco > 0 | 0/112650 linhas falharam


order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,_ingested_at
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.9,13.29,2026-09-15T01:07:02.330Z
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.9,19.93,2026-09-15T01:07:02.330Z
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.0,17.87,2026-09-15T01:07:02.330Z
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79,2026-09-15T01:07:02.330Z
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.9,18.14,2026-09-15T01:07:02.330Z
00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23T03:55:27.000Z,21.9,12.69,2026-09-15T01:07:02.330Z
00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14T12:10:31.000Z,19.9,11.85,2026-09-15T01:07:02.330Z
000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10T12:30:45.000Z,810.0,70.75,2026-09-15T01:07:02.330Z
0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26T18:31:29.000Z,145.95,11.65,2026-09-15T01:07:02.330Z
0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06T14:10:56.000Z,53.99,11.4,2026-09-15T01:07:02.330Z


id_pedido,id_item,id_produto,id_vendedor,data_limite_envio,preco,preco_frete
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.90,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.90,19.93
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.00,17.87
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.90,18.14
00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23T03:55:27.000Z,21.90,12.69
00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14T12:10:31.000Z,19.90,11.85
000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10T12:30:45.000Z,810.00,70.75
0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26T18:31:29.000Z,145.95,11.65
0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06T14:10:56.000Z,53.99,11.40


## 5. order_payments
- Deduplicação por `(order_id, payment_sequential)` → `(id_pedido, codigo_pagamento)`
- Tradução de `payment_type` para português (credit_card→Cartão de Crédito, boleto→Boleto, etc.)
- Mapeamento: `payment_sequential`→`codigo_pagamento`, `payment_type`→`forma_pagamento`, `payment_installments`→`parcelas`
- Validação: `valor_pagamento` não pode ser negativo

In [0]:
df_bronze_payments = spark.table(f"{bronze_schema}.order_payments")

df_silver_payments = (
    df_bronze_payments
    .dropDuplicates(["order_id", "payment_sequential"])
    .filter(F.col("order_id").isNotNull())
    .withColumn("payment_type", F.lower(F.trim(F.col("payment_type"))))
    .withColumn("payment_type",
        F.when(F.col("payment_type") == "credit_card", "Cartão de Crédito")
        .when(F.col("payment_type") == "boleto", "Boleto")
        .when(F.col("payment_type") == "voucher", "Voucher")
        .when(F.col("payment_type") == "debit_card", "Cartão de Débito")
        .otherwise("Outro")
    )
    .withColumn("payment_value", F.col("payment_value").cast("decimal(10,2)"))
    .filter(F.col("payment_value") >= 0)
    .select("order_id", "payment_sequential", "payment_type",
            "payment_installments", "payment_value")
    .withColumnRenamed("order_id", "id_pedido")
    .withColumnRenamed("payment_sequential", "codigo_pagamento")
    .withColumnRenamed("payment_type", "forma_pagamento")
    .withColumnRenamed("payment_installments", "parcelas")
    .withColumnRenamed("payment_value", "valor_pagamento")
)

dq_check("silver.order_payments", "valor_pagamento >= 0", df_silver_payments,
         F.col("valor_pagamento") >= 0)

df_silver_payments.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.order_payments")
print("silver.order_payments gravada com sucesso.")

display(df_bronze_payments.limit(10))
display(df_silver_payments.limit(10))

[PASS] silver.order_payments | valor_pagamento >= 0 | 0/103886 linhas falharam
silver.order_payments gravada com sucesso.


order_id,payment_sequential,payment_type,payment_installments,payment_value,_ingested_at
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33,2026-09-15T01:07:06.955Z
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39,2026-09-15T01:07:06.955Z
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71,2026-09-15T01:07:06.955Z
ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78,2026-09-15T01:07:06.955Z
42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45,2026-09-15T01:07:06.955Z
298fcdf1f73eb413e4d26d01b25bc1cd,1,credit_card,2,96.12,2026-09-15T01:07:06.955Z
771ee386b001f06208a7419e4fc1bbd7,1,credit_card,1,81.16,2026-09-15T01:07:06.955Z
3d7239c394a212faae122962df514ac7,1,credit_card,3,51.84,2026-09-15T01:07:06.955Z
1f78449c87a54faf9e96e88ba1491fa9,1,credit_card,6,341.09,2026-09-15T01:07:06.955Z
0573b5e23cbd798006520e1d5b4c6714,1,boleto,1,51.95,2026-09-15T01:07:06.955Z


id_pedido,codigo_pagamento,forma_pagamento,parcelas,valor_pagamento
b81ef226f3fe1789b1e8b2acac839d17,1,Cartão de Crédito,8,99.33
a9810da82917af2d9aefd1278f1dcfa0,1,Cartão de Crédito,1,24.39
25e8ea4e93396b6fa0d3dd708e76c1bd,1,Cartão de Crédito,1,65.71
ba78997921bbcdc1373bb41e913ab953,1,Cartão de Crédito,8,107.78
42fdf880ba16b47b59251dd489d4441a,1,Cartão de Crédito,2,128.45
298fcdf1f73eb413e4d26d01b25bc1cd,1,Cartão de Crédito,2,96.12
771ee386b001f06208a7419e4fc1bbd7,1,Cartão de Crédito,1,81.16
3d7239c394a212faae122962df514ac7,1,Cartão de Crédito,3,51.84
1f78449c87a54faf9e96e88ba1491fa9,1,Cartão de Crédito,6,341.09
0573b5e23cbd798006520e1d5b4c6714,1,Boleto,1,51.95


In [0]:
df_dq_log = spark.createDataFrame(dq_results)
df_dq_log.write.format("delta").mode("append").option("mergeSchema", "true") \
    .saveAsTable(f"{silver_schema}.dq_log")

display(spark.table(f"{silver_schema}.dq_log").orderBy(F.col("checked_at").desc()))

table_name,check_name,total_rows,failed_rows,passed,checked_at
silver.order_payments,valor_pagamento >= 0,103886,0,true,2026-09-15T02:48:10.106Z
silver.order_items,preco > 0,112650,0,true,2026-09-15T02:48:04.029Z
silver.order_items,"(id_pedido, id_item) único",112650,0,true,2026-09-15T02:48:02.868Z
silver.products,peso_produto_gramas não negativo,32951,0,true,2026-09-15T02:47:56.002Z
silver.products,id_produto único,32951,0,true,2026-09-15T02:47:54.073Z
silver.orders,status_pedido em domínio válido,99441,0,true,2026-09-15T02:47:46.397Z
silver.orders,data_compra_pedido não nulo,99441,0,true,2026-09-15T02:47:44.903Z
silver.orders,id_pedido único,99441,0,true,2026-09-15T02:47:43.240Z
silver.customers,estado_cliente com 2 caracteres,99441,0,true,2026-09-15T02:47:36.328Z
silver.customers,id_consumidor único,99441,0,true,2026-09-15T02:47:35.172Z
